DeNovoSeer

#Xiyu Rao
#2026-04-20

In [ ]:
"""
SHAP analysis for DeNovoSeer – Jupyter Notebook version
Based on shap_analysis_denovoseer.py with hardcoded paths.
"""

import os
import json
import warnings
from typing import List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

try:
    import shap
except ImportError as e:
    raise ImportError("Please install shap: pip install shap") from e

# =========================
# Model definition (same as training)
# =========================
class SemiSupervisedHybridCNNIDCNN(nn.Module):
    def __init__(self, input_channels=1, num_classes=2, feature_dim=305):
        super().__init__()
        self.feature_dim = feature_dim
        self.input_channels = input_channels

        self.feature_extractor = nn.Sequential(
            nn.Conv1d(input_channels, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1, dilation=1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=4, dilation=4),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, 256, kernel_size=3, padding=8, dilation=8),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )
        self.reconstructor = nn.Sequential(
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, input_channels * feature_dim),
        )
        self.projection_head = nn.Sequential(
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
        )

    def forward(self, x, mode="supervised"):
        features = self.feature_extractor(x)
        if mode == "supervised":
            return self.classifier(features)
        if mode == "reconstruction":
            return self.reconstructor(features)
        if mode == "contrastive":
            return self.projection_head(features)
        if mode == "features":
            return features
        raise ValueError("Invalid mode")

class PathogenicProbWrapper(nn.Module):
    def __init__(self, base_model: nn.Module):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        logits = self.base_model(x, mode="supervised")
        probs = F.softmax(logits, dim=1)[:, 1]
        return probs.unsqueeze(1)

# =========================
# Utility functions
# =========================
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def load_processed_data(data_path: str):
    df = pd.read_csv(data_path)
    non_numeric_columns = [
        "ClinVar_label", "variant_index", "Otherinfo", "Phenotype",
        "platform", "study", "PMID", "Explanation"
    ]
    available_non_numeric = [col for col in non_numeric_columns if col in df.columns]
    numeric_feature_names = [
        col for col in df.columns if col not in available_non_numeric and col != "label"
    ]
    labels_all = df["label"].values
    features_all = df[numeric_feature_names].values.astype(np.float32)
    non_numeric_all = df[available_non_numeric].copy() if available_non_numeric else pd.DataFrame(index=df.index)
    if np.isnan(features_all).sum() > 0:
        raise ValueError("NaN in features_all")
    return df, features_all, labels_all, non_numeric_all, numeric_feature_names, available_non_numeric

def rebuild_split_indices(labels_all: np.ndarray, seed: int):
    indices = np.arange(len(labels_all))
    train_idx, temp_idx, train_y, temp_y = train_test_split(
        indices, labels_all, test_size=0.4, stratify=labels_all, random_state=seed
    )
    val_idx, test_idx, _, _ = train_test_split(
        temp_idx, temp_y, test_size=0.5, stratify=temp_y, random_state=seed
    )
    return train_idx, val_idx, test_idx

def validate_test_index(saved_test_idx, rebuilt_test_idx):
    if len(saved_test_idx) != len(rebuilt_test_idx):
        raise ValueError("Test index length mismatch")
    if not np.array_equal(saved_test_idx, rebuilt_test_idx):
        raise ValueError("Test index mismatch")

@torch.no_grad()
def predict_probabilities(base_model, x_np, device, batch_size=256):
    base_model.eval()
    probs = []
    n = len(x_np)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = torch.tensor(x_np[start:end], dtype=torch.float32, device=device).unsqueeze(1)
        logits = base_model(batch, mode="supervised")
        batch_probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
        probs.append(batch_probs)
    return np.concatenate(probs, axis=0)

@torch.no_grad()
def predict_labels(base_model, x_np, device, batch_size=256):
    base_model.eval()
    preds = []
    n = len(x_np)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch = torch.tensor(x_np[start:end], dtype=torch.float32, device=device).unsqueeze(1)
        logits = base_model(batch, mode="supervised")
        batch_preds = torch.argmax(logits, dim=1).cpu().numpy()
        preds.append(batch_preds)
    return np.concatenate(preds, axis=0)

def compute_shap_values(prob_model, background_np, explain_np, device):
    background_tensor = torch.tensor(background_np, dtype=torch.float32, device=device).unsqueeze(1)
    explain_tensor = torch.tensor(explain_np, dtype=torch.float32, device=device).unsqueeze(1)
    explainer_name = None
    try:
        explainer = shap.DeepExplainer(prob_model, background_tensor)
        shap_values = explainer.shap_values(explain_tensor)
        expected_value = explainer.expected_value
        explainer_name = "DeepExplainer"
    except Exception as e:
        print(f"[Warning] DeepExplainer failed: {e}")
        print("Falling back to GradientExplainer...")
        explainer = shap.GradientExplainer(prob_model, background_tensor)
        shap_values = explainer.shap_values(explain_tensor)
        expected_value = None
        explainer_name = "GradientExplainer"
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    shap_values = np.array(shap_values)
    if shap_values.ndim == 3 and shap_values.shape[1] == 1:
        shap_values = shap_values[:, 0, :]
    elif shap_values.ndim == 3 and shap_values.shape[2] == 1:
        shap_values = shap_values[:, :, 0]
    if expected_value is not None:
        if isinstance(expected_value, list):
            expected_value = expected_value[0]
        if isinstance(expected_value, np.ndarray):
            expected_value = float(np.ravel(expected_value)[0])
    return explainer, shap_values, expected_value, explainer_name

def save_global_outputs(shap_values, x_explain_df, output_dir, max_display=20):
    feature_names = list(x_explain_df.columns)
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    importance_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False)
    importance_df.to_csv(os.path.join(output_dir, "global_shap_importance.csv"), index=False)
    plt.figure()
    shap.summary_plot(shap_values, x_explain_df, feature_names=feature_names,
                      max_display=max_display, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"global_shap_summary_top{max_display}.png"), dpi=300, bbox_inches="tight")
    plt.close()
    plt.figure()
    shap.summary_plot(shap_values, x_explain_df, feature_names=feature_names,
                      plot_type="bar", max_display=max_display, show=False)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"global_shap_bar_top{max_display}.png"), dpi=300, bbox_inches="tight")
    plt.close()
    return importance_df

def build_case_selection(case_mode, n_cases, test_meta_df, test_labels, pred_probs, pred_labels):
    indices = []
    all_idx = np.arange(len(test_labels))
    if case_mode == "auto":
        path_idx = all_idx[test_labels == 1]
        if len(path_idx) > 0:
            chosen = path_idx[np.argsort(-pred_probs[path_idx])[: max(1, n_cases // 3 + 1)]]
            indices.extend(chosen.tolist())
        benign_idx = all_idx[test_labels == 0]
        if len(benign_idx) > 0:
            chosen = benign_idx[np.argsort(pred_probs[benign_idx])[: max(1, n_cases // 3 + 1)]]
            indices.extend(chosen.tolist())
        labeled_idx = all_idx[test_labels != -1]
        if len(labeled_idx) > 0:
            dist = np.abs(pred_probs[labeled_idx] - 0.5)
            chosen = labeled_idx[np.argsort(dist)[: max(1, n_cases // 3 + 1)]]
            indices.extend(chosen.tolist())
        final_indices = []
        seen = set()
        for idx in indices:
            if idx not in seen:
                final_indices.append(int(idx))
                seen.add(int(idx))
            if len(final_indices) >= n_cases:
                break
        return final_indices
    raise ValueError(f"Unsupported case_mode: {case_mode}")

def save_local_case_outputs(shap_values, expected_value, x_explain_df, case_indices,
                            test_meta_df, test_labels, pred_probs, pred_labels,
                            output_dir, max_display=15):
    feature_names = list(x_explain_df.columns)
    case_records = []
    for rank, idx in enumerate(case_indices, start=1):
        case_prefix = f"case_{rank}_idx_{idx}"
        x_row = x_explain_df.iloc[idx]
        sv_row = shap_values[idx]
        local_df = pd.DataFrame({
            "feature": feature_names,
            "feature_value": x_row.values,
            "shap_value": sv_row,
        })
        local_df["abs_shap"] = local_df["shap_value"].abs()
        local_df = local_df.sort_values("abs_shap", ascending=False)
        local_df.to_csv(os.path.join(output_dir, f"{case_prefix}_local_shap.csv"), index=False)
        meta_info = test_meta_df.iloc[idx].to_dict() if not test_meta_df.empty else {}
        meta_info.update({
            "case_idx_in_testset": int(idx),
            "true_label": int(test_labels[idx]),
            "pred_label": int(pred_labels[idx]),
            "pred_prob": float(pred_probs[idx]),
        })
        with open(os.path.join(output_dir, f"{case_prefix}_meta.json"), "w", encoding="utf-8") as f:
            json.dump(meta_info, f, ensure_ascii=False, indent=2)
        try:
            base_val = expected_value if expected_value is not None else 0.0
            explanation = shap.Explanation(
                values=sv_row,
                base_values=base_val,
                data=x_row.values,
                feature_names=feature_names,
            )
            plt.figure()
            shap.plots.waterfall(explanation, max_display=max_display, show=False)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"{case_prefix}_waterfall.png"), dpi=300, bbox_inches="tight")
            plt.close()
        except Exception as e:
            print(f"Warning: waterfall failed for {case_prefix}: {e}")
        try:
            base_val = expected_value if expected_value is not None else 0.0
            force = shap.force_plot(base_val, sv_row, x_row.values,
                                    feature_names=feature_names, matplotlib=False)
            shap.save_html(os.path.join(output_dir, f"{case_prefix}_force.html"), force)
        except Exception as e:
            print(f"Warning: force plot failed for {case_prefix}: {e}")
        case_records.append({
            "case_rank": rank,
            "case_idx_in_testset": int(idx),
            "true_label": int(test_labels[idx]),
            "pred_label": int(pred_labels[idx]),
            "pred_prob": float(pred_probs[idx]),
            "top_features": "; ".join(local_df.head(10)["feature"].tolist()),
        })
    pd.DataFrame(case_records).to_csv(os.path.join(output_dir, "selected_case_summary.csv"), index=False)

def save_test_prediction_table(test_meta_df, test_labels, pred_labels, pred_probs, output_dir):
    out_df = test_meta_df.copy() if not test_meta_df.empty else pd.DataFrame(index=np.arange(len(test_labels)))
    out_df["true_label"] = test_labels
    out_df["pred_label"] = pred_labels
    out_df["pred_prob"] = pred_probs
    out_df.to_csv(os.path.join(output_dir, "test_set_predictions_with_metadata.csv"), index=False)

# =========================
# Main execution (hardcoded paths)
# =========================
def main():
    # ====== HARDCODED PARAMETERS (adjust as needed) ======
    # replace with your actual data_path
    data_path = "processed_coding_data_scaled.csv"
    # replace with your actual model_path
    model_path = "best_model_seed42.pth"
    # replace with your actual test_index_path
    test_index_path = "coding_test_indices_seed42.npy" 
    seed = 42
    output_dir = "shap_outputs_seed42"
    background_size = 200
    max_explain_samples = 1000
    global_max_display = 20
    local_max_display = 15
    n_cases = 3
    case_mode = "auto"
    # =====================================================

    os.makedirs(output_dir, exist_ok=True)
    set_seed(seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    print("Loading processed data...")
    df, features_all, labels_all, non_numeric_all, feature_names, meta_cols = load_processed_data(data_path)
    print(f"Total samples: {len(df)}, feature dim: {len(feature_names)}")

    print("Rebuilding train/val/test split...")
    train_idx, val_idx, rebuilt_test_idx = rebuild_split_indices(labels_all, seed=seed)
    saved_test_idx = np.load(test_index_path)
    validate_test_index(saved_test_idx, rebuilt_test_idx)
    print("Test index validated.")

    test_idx = saved_test_idx
    train_feats = features_all[train_idx]
    test_feats = features_all[test_idx]
    test_labels = labels_all[test_idx]
    test_meta = non_numeric_all.iloc[test_idx].copy().reset_index(drop=True)

    # Use only labeled test samples
    labeled_mask = test_labels != -1
    test_feats_labeled = test_feats[labeled_mask]
    test_labels_labeled = test_labels[labeled_mask]
    test_meta_labeled = test_meta.iloc[np.where(labeled_mask)[0]].reset_index(drop=True)

    if len(test_feats_labeled) == 0:
        raise ValueError("No labeled samples in test set.")
    if len(test_feats_labeled) > max_explain_samples:
        keep = np.arange(len(test_feats_labeled))[:max_explain_samples]
        test_feats_labeled = test_feats_labeled[keep]
        test_labels_labeled = test_labels_labeled[keep]
        test_meta_labeled = test_meta_labeled.iloc[keep].reset_index(drop=True)

    background_size = min(background_size, len(train_feats))
    bg_indices = np.random.choice(len(train_feats), size=background_size, replace=False)
    background_feats = train_feats[bg_indices]

    print(f"Background size: {len(background_feats)}")
    print(f"Explain sample size (labeled test): {len(test_feats_labeled)}")

    print("Loading model...")
    model = SemiSupervisedHybridCNNIDCNN(
        input_channels=1,
        num_classes=2,
        feature_dim=features_all.shape[1],
    ).to(device)
    state = torch.load(model_path, map_location=device)
    model.load_state_dict(state)
    model.eval()

    prob_model = PathogenicProbWrapper(model).to(device)
    prob_model.eval()

    print("Predicting probabilities...")
    pred_probs = predict_probabilities(model, test_feats_labeled, device=device)
    pred_labels = (pred_probs >= 0.5).astype(int)

    save_test_prediction_table(test_meta_labeled, test_labels_labeled, pred_labels, pred_probs, output_dir)

    print("Computing SHAP values...")
    explainer, shap_values, expected_value, explainer_name = compute_shap_values(
        prob_model, background_feats, test_feats_labeled, device
    )
    print(f"Explainer: {explainer_name}, SHAP shape: {shap_values.shape}")

    x_explain_df = pd.DataFrame(test_feats_labeled, columns=feature_names)

    print("Saving global SHAP outputs...")
    importance_df = save_global_outputs(shap_values, x_explain_df, output_dir, max_display=global_max_display)
    print("Top 20 features:\n", importance_df.head(20).to_string(index=False))

    print("Selecting local cases...")
    case_indices = build_case_selection(
        case_mode, n_cases, test_meta_labeled, test_labels_labeled, pred_probs, pred_labels
    )
    print("Selected case indices:", case_indices)

    print("Saving local case outputs...")
    save_local_case_outputs(
        shap_values, expected_value, x_explain_df, case_indices,
        test_meta_labeled, test_labels_labeled, pred_probs, pred_labels,
        output_dir, max_display=local_max_display
    )

    run_info = {
        "seed": seed,
        "data_path": data_path,
        "model_path": model_path,
        "test_index_path": test_index_path,
        "background_size": int(background_size),
        "explain_sample_size": int(len(test_feats_labeled)),
        "explainer_name": explainer_name,
        "expected_value": None if expected_value is None else float(expected_value),
        "feature_dim": int(len(feature_names)),
        "meta_columns": meta_cols,
        "selected_case_indices": case_indices,
    }
    with open(os.path.join(output_dir, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(run_info, f, ensure_ascii=False, indent=2)

    print(f"Done. Outputs saved in: {output_dir}")

if __name__ == "__main__":
    main()